In [10]:
import requests
find = "ark"
# Find Arkansas's ESPN team ID
r = requests.get(
    "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams?limit=500"
).json()

teams = [(t["team"]["id"], t["team"]["displayName"]) 
         for t in r["sports"][0]["leagues"][0]["teams"]]

teams = [t for t in teams if find in t[1].lower()]
print(teams)

ConnectionError: HTTPSConnectionPool(host='site.api.espn.com', port=443): Max retries exceeded with url: /apis/site/v2/sports/basketball/mens-college-basketball/teams?limit=500 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000130FDE9FB10>: Failed to resolve 'site.api.espn.com' ([Errno 11001] getaddrinfo failed)"))

In [4]:
import requests
import pandas as pd

# ── CONFIG ───────────────────────────────────────────────────────────────────
TEAM_NAME1 = "Arkansas"
TEAM_ID1  = 8

TEAM_NAME2 = "Arkansas"
TEAM_ID2  = 8

SEASON    = 2026
# ─────────────────────────────────────────────────────────────────────────────

# 1. Pull ALL games across all season types (1=pre, 2=regular, 3=post)
all_schedule_rows = []
for season_type in [1, 2, 3]:
    url  = (f"https://site.api.espn.com/apis/site/v2/sports/basketball/"
            f"mens-college-basketball/teams/{TEAM_ID}/schedule"
            f"?season={SEASON}&seasontype={season_type}")
    resp = requests.get(url).json()

    for g in resp.get("events", []):
        comp      = g["competitions"][0]
        completed = comp["status"]["type"]["completed"]
        opponent  = next(
            t["team"]["displayName"]
            for t in comp["competitors"]
            if t["team"]["id"] != str(TEAM_ID)
        )
        all_schedule_rows.append({
            "game_id"    : g["id"],
            "date"       : g["date"],
            "opponent"   : opponent,
            "season_type": season_type,
            "completed"  : completed
        })

schedule_df = pd.DataFrame(all_schedule_rows)
game_ids    = schedule_df[schedule_df["completed"]]["game_id"].tolist()

print(f"Found {len(game_ids)} completed games\n")
print(schedule_df.to_string(index=False))

# 2. Pull boxscores via ESPN API
all_rows = []
for gid in game_ids:
    try:
        url  = (f"https://site.api.espn.com/apis/site/v2/sports/basketball/"
                f"mens-college-basketball/summary?event={gid}")
        data = requests.get(url).json()

        for team_box in data.get("boxscore", {}).get("players", []):
            team_name = team_box["team"]["displayName"]
            if TEAM_NAME.lower() not in team_name.lower():
                continue

            labels = [s["name"] for s in team_box["statistics"][0]["labels"]]

            for athlete in team_box["statistics"][0]["athletes"]:
                stats            = dict(zip(labels, athlete["stats"]))
                stats["player"]  = athlete["athlete"]["displayName"]
                stats["game_id"] = gid
                all_rows.append(stats)

    except Exception as e:
        print(f"  Skipping {gid}: {e}")

raw_df = pd.concat([pd.DataFrame([r]) for r in all_rows], ignore_index=True)
print(f"\nColumns available: {raw_df.columns.tolist()}")

# 3. Rename ESPN columns to friendly names
col_map = {
    "PTS": "pts", "REB": "reb", "AST": "ast",
    "STL": "stl", "BLK": "blk", "TO":  "to",
    "FGM-FGA": "fg", "3PM-3PA": "3p", "FTM-FTA": "ft",
    "OREB": "oreb", "DREB": "dreb", "PF": "pf", "MIN": "min"
}
raw_df.rename(columns=col_map, inplace=True)

# Split "made-att" columns into separate made/att columns
for stat, made, att in [("fg", "fgm", "fga"), ("3p", "3pm", "3pa"), ("ft", "ftm", "fta")]:
    if stat in raw_df.columns:
        raw_df[[made, att]] = raw_df[stat].str.split("-", expand=True)

stat_cols = ["pts", "reb", "ast", "stl", "blk", "to", "pf",
             "oreb", "dreb", "fgm", "fga", "3pm", "3pa", "ftm", "fta"]
stat_cols = [c for c in stat_cols if c in raw_df.columns]
raw_df[stat_cols] = raw_df[stat_cols].apply(pd.to_numeric, errors="coerce")

# 4. Aggregate season totals
totals = raw_df.groupby("player", as_index=False)[stat_cols].sum()
gp     = (raw_df.groupby("player", as_index=False)
          .size()
          .rename(columns={"size": "games_played"}))
season_stats = totals.merge(gp, on="player")

# 5. Per-game averages
for col in ["pts", "reb", "ast", "stl", "blk"]:
    if col in season_stats.columns:
        season_stats[f"{col}_pg"] = (
            season_stats[col].astype(float) / season_stats["games_played"]
        ).round(1)

# 6. Shooting percentages
def safe_pct(made, att, df):
    return (df[made].astype(float) /
            df[att].astype(float).where(df[att] > 0) * 100).round(1)

season_stats["fg_pct"] = safe_pct("fgm", "fga", season_stats)
season_stats["3p_pct"] = safe_pct("3pm", "3pa", season_stats)
season_stats["ft_pct"] = safe_pct("ftm", "fta", season_stats)

# 7. Display & export
season_stats = season_stats.sort_values("pts_pg", ascending=False).reset_index(drop=True)
print(season_stats[["player", "games_played",
                     "pts_pg", "reb_pg", "ast_pg",
                     "fg_pct", "3p_pct", "ft_pct"]].to_string(index=False))

Found 34 completed games

  game_id              date                   opponent  season_type  completed
401826784 2025-11-04T00:00Z           Southern Jaguars            2       True
401826785 2025-11-09T00:00Z    Michigan State Spartans            2       True
401824935 2025-11-12T01:00Z     Central Arkansas Bears            2       True
401826786 2025-11-15T01:00Z           Samford Bulldogs            2       True
401826787 2025-11-19T01:00Z            Winthrop Eagles            2       True
401826788 2025-11-22T01:00Z       Jackson State Tigers            2       True
401817234 2025-11-28T01:00Z           Duke Blue Devils            2       True
401806374 2025-12-04T00:15Z       Louisville Cardinals            2       True
401826791 2025-12-06T21:00Z      Fresno State Bulldogs            2       True
401826789 2025-12-13T17:00Z     Texas Tech Red Raiders            2       True
401823497 2025-12-17T02:00Z   Queens University Royals            2       True
401820294 2025-12-20T22:30

ValueError: No objects to concatenate